# BERT на DUSHA: Whisper ASR → rubert-tiny2 fine-tune

Pipeline:
1. Аудио DUSHA → текст через `artyomboyko/whisper-small-ru-v2`
2. Текст → fine-tune `cointegrated/rubert-tiny2-cedr-emotion-detection` (5 классов DUSHA)

Val (5000 сэмплов) транскрибируется **один раз** перед обучением и кешируется в памяти.  
Train транскрибируется тоже один раз (батчами) и кешируется.  
Eval каждые **2500 шагов** по val loss.  
Лучшая модель сохраняется в `/kaggle/working/`.

## 1. Install & clone

In [ ]:
import subprocess, sys, os

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'datasets', 'soundfile', 'torchaudio',
    'pyyaml', 'tqdm', 'accelerate',
], check=True)

REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Done. CWD:', os.getcwd())

## 2. Imports & config

In [ ]:
import warnings, pathlib, random, json
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoProcessor, AutoModelForSpeechSeq2Seq,
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# ── пути ──────────────────────────────────────────────────────────────────────
AGG_ROOT         = pathlib.Path('/kaggle/input/datasets/aleksandribryanov/agg-dusha')
AUDIO_TRAIN_DIR  = pathlib.Path('/kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd/crowd_train')
AUDIO_TEST_DIR   = pathlib.Path('/kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd/crowd_test')
TRAIN_TSV        = AGG_ROOT / 'aggregated_majority.tsv'
TEST_TSV         = AGG_ROOT / 'aggregated_ds_0.9_test.tsv'
OUT_DIR          = pathlib.Path('/kaggle/working')

# ── гиперпараметры ────────────────────────────────────────────────────────────
TRAIN_FRACTION  = 0.05    # доля train TSV (~7k записей)
VAL_SIZE        = 5000    # сэмплов для валидации
WHISPER_BATCH   = 16      # батч для ASR транскрипции
BERT_BATCH      = 32      # батч для BERT обучения
LR              = 2e-5    # маленький LR для full fine-tune
WEIGHT_DECAY    = 1e-2
WARMUP_STEPS    = 200
MAX_STEPS       = 20_000
EVAL_EVERY      = 2500    # шагов между валидациями
ES_PATIENCE     = 5       # early stopping: кол-во eval без улучшения
MAX_SEQ_LEN     = 128
SR_TARGET       = 16_000
SEED            = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DUSHA_LABEL2ID = {'neutral': 0, 'angry': 1, 'positive': 2, 'sad': 3, 'other': 4}
DUSHA_LABELS   = ['neutral', 'angry', 'positive', 'sad', 'other']

## 3. Загрузка DUSHA

In [ ]:
def load_tsv_records(tsv_path, audio_dir, fraction=None):
    df = pd.read_csv(tsv_path, sep='\t')
    df = df[df['aggregated_emo'].isin(DUSHA_LABEL2ID)]
    if fraction:
        df = df.sample(frac=fraction, random_state=SEED)
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'Loading {tsv_path.name}', leave=False):
        path = audio_dir / row['audio_path']
        if path.exists():
            records.append({
                'path':  str(path),
                'label': DUSHA_LABEL2ID[row['aggregated_emo']],
            })
    return records

all_train = load_tsv_records(TRAIN_TSV, AUDIO_TRAIN_DIR, fraction=TRAIN_FRACTION)

# выделяем val_size сэмплов для валидации (стратифицированно)
from sklearn.model_selection import train_test_split
train_recs, val_recs = train_test_split(
    all_train, test_size=VAL_SIZE, random_state=SEED,
    stratify=[r['label'] for r in all_train],
)
print(f'Train: {len(train_recs)}  Val: {len(val_recs)}')
from collections import Counter
for lid, name in enumerate(DUSHA_LABELS):
    t = sum(1 for r in train_recs if r['label'] == lid)
    v = sum(1 for r in val_recs   if r['label'] == lid)
    print(f'  {name:10s}  train={t:4d}  val={v:4d}')

## 4. Whisper ASR

In [ ]:
import librosa

WHISPER_MODEL = 'artyomboyko/whisper-small-ru-v2'

print(f'Loading Whisper: {WHISPER_MODEL}')
asr_processor = AutoProcessor.from_pretrained(WHISPER_MODEL)
asr_model     = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)
asr_model.eval()
print('Whisper loaded.')


def load_wav(path):
    wav, sr = sf.read(path, dtype='float32')
    if wav.ndim > 1: wav = wav.mean(axis=1)
    if sr != SR_TARGET:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
    return wav


@torch.no_grad()
def transcribe_records(records, batch_size=WHISPER_BATCH, desc='ASR'):
    texts = []
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    for i in tqdm(range(0, len(records), batch_size), desc=desc):
        batch  = records[i:i + batch_size]
        wavs   = [load_wav(r['path']) for r in batch]
        inputs = asr_processor(wavs, sampling_rate=SR_TARGET, return_tensors='pt', padding=True)
        ids    = asr_model.generate(
            inputs.input_features.to(device, dtype=dtype),
            language='ru', task='transcribe',
        )
        texts.extend(asr_processor.batch_decode(ids, skip_special_tokens=True))
    return texts

### 4.1 Smoke test — 5 сэмплов

In [ ]:
from IPython.display import display, Audio

smoke_recs = random.sample(all_train, 5)
smoke_texts = transcribe_records(smoke_recs, batch_size=5, desc='Smoke ASR')

print(f'{"#":<3}  {"Emotion":<10}  Transcription')
print('-' * 70)
for i, (rec, text) in enumerate(zip(smoke_recs, smoke_texts)):
    label_name = DUSHA_LABELS[rec['label']]
    print(f'{i+1:<3}  {label_name:<10}  {text}')
    wav = load_wav(rec['path'])
    display(Audio(wav, rate=SR_TARGET))

### 4.2 Транскрипция val + train (один раз)

In [ ]:
print('Transcribing val set (one-time)...')
val_texts  = transcribe_records(val_recs,   desc='Val ASR')
val_labels = [r['label'] for r in val_recs]
print(f'Val done. Example: "{val_texts[0]}"')

print('Transcribing train set (one-time)...')
train_texts  = transcribe_records(train_recs, desc='Train ASR')
train_labels = [r['label'] for r in train_recs]
print(f'Train done. {len(train_texts)} texts.')

del asr_model, asr_processor
torch.cuda.empty_cache()
print('Whisper unloaded from GPU.')

## 5. BERT модель

In [ ]:
BERT_MODEL = 'cointegrated/rubert-tiny2-cedr-emotion-detection'

print(f'Loading BERT: {BERT_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL,
    num_labels=len(DUSHA_LABELS),
    ignore_mismatched_sizes=True,   # заменяем голову на 5 классов DUSHA
).to(device)

total = sum(p.numel() for p in bert.parameters())
print(f'Parameters: {total:,}')
print('Label mapping:', {i: l for i, l in enumerate(DUSHA_LABELS)})


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_SEQ_LEN):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {
            'input_ids':      self.encodings['input_ids'][i],
            'attention_mask': self.encodings['attention_mask'][i],
            'labels':         self.labels[i],
        }


print('Tokenizing...')
train_ds = TextDataset(train_texts, train_labels, tokenizer)
val_ds   = TextDataset(val_texts,   val_labels,   tokenizer)
train_ld = DataLoader(train_ds, batch_size=BERT_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_ld   = DataLoader(val_ds,   batch_size=BERT_BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_ld)}  Val batches: {len(val_ld)}')

## 6. Обучение

In [ ]:
from sklearn.metrics import balanced_accuracy_score

optimizer = torch.optim.AdamW(bert.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_warmup = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=MAX_STEPS,
)
scheduler_plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

best_val_loss = float('inf')
es_counter    = 0
global_step   = 0
history       = []   # (step, train_loss, val_loss, val_wacc)
train_iter    = iter(train_ld)


@torch.no_grad()
def evaluate():
    bert.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in val_ld:
        batch = {k: v.to(device) for k, v in batch.items()}
        out   = bert(**batch)
        total_loss  += out.loss.item() * batch['labels'].size(0)
        preds_all.append(out.logits.argmax(-1).cpu().numpy())
        labels_all.append(batch['labels'].cpu().numpy())
    val_loss = total_loss / len(val_ds)
    val_wacc = balanced_accuracy_score(
        np.concatenate(labels_all), np.concatenate(preds_all)
    )
    bert.train()
    return val_loss, val_wacc


print(f'Starting training  max_steps={MAX_STEPS}  eval_every={EVAL_EVERY}  es_patience={ES_PATIENCE}')
print(f'LR={LR}  batch={BERT_BATCH}  warmup={WARMUP_STEPS}')
print('-' * 70)

bert.train()
running_loss = 0.0

while global_step < MAX_STEPS:
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_ld)
        batch = next(train_iter)

    batch = {k: v.to(device) for k, v in batch.items()}
    optimizer.zero_grad()
    out  = bert(**batch)
    loss = out.loss
    loss.backward()
    nn.utils.clip_grad_norm_(bert.parameters(), 1.0)
    optimizer.step()
    scheduler_warmup.step()
    running_loss += loss.item()
    global_step  += 1

    if global_step % EVAL_EVERY == 0:
        avg_train_loss = running_loss / EVAL_EVERY
        val_loss, val_wacc = evaluate()
        scheduler_plateau.step(val_loss)
        running_loss = 0.0

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss = val_loss
            es_counter    = 0
            bert.save_pretrained(str(OUT_DIR / 'best_bert_dusha'))
            tokenizer.save_pretrained(str(OUT_DIR / 'best_bert_dusha'))
        else:
            es_counter += 1

        lr_cur = optimizer.param_groups[0]['lr']
        history.append((global_step, avg_train_loss, val_loss, val_wacc))
        print(
            f'Step {global_step:6d}  train_loss={avg_train_loss:.4f}  '
            f'val_loss={val_loss:.4f}  val_wacc={val_wacc:.4f}  '
            f'lr={lr_cur:.2e}' + ('  *' if is_best else ''),
            flush=True,
        )

        if es_counter >= ES_PATIENCE:
            print(f'Early stopping at step {global_step} (no improvement for {ES_PATIENCE} evals)')
            break

print(f'\nBest val_loss: {best_val_loss:.4f}')
print(f'Model saved → {OUT_DIR / "best_bert_dusha"}')

## 7. Кривые обучения

In [ ]:
import matplotlib.pyplot as plt

steps      = [h[0] for h in history]
tr_losses  = [h[1] for h in history]
val_losses = [h[2] for h in history]
val_waccs  = [h[3] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(steps, tr_losses,  label='train loss', color='steelblue')
ax1.plot(steps, val_losses, label='val loss',   color='darkorange')
best_step = steps[int(np.argmin(val_losses))]
ax1.axvline(best_step, color='red', linestyle='--', alpha=0.6, label=f'best step {best_step}')
ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
ax1.set_title('Train / Val Loss'); ax1.legend()

ax2.plot(steps, val_waccs, color='green', label='val wacc')
ax2.axvline(best_step, color='red', linestyle='--', alpha=0.6)
ax2.set_xlabel('Step'); ax2.set_ylabel('Weighted Accuracy')
ax2.set_title('Val Weighted Accuracy'); ax2.legend()

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'bert_training_curves.png'), dpi=150)
plt.show()

## 8. Финальная оценка на test TSV

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# загружаем сохранённую модель
bert_best = AutoModelForSequenceClassification.from_pretrained(
    str(OUT_DIR / 'best_bert_dusha')).to(device)
bert_best.eval()

# транскрибируем тест
print('Loading Whisper for test transcription...')
asr_processor = AutoProcessor.from_pretrained(WHISPER_MODEL)
asr_model     = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)
asr_model.eval()

test_recs  = load_tsv_records(TEST_TSV, AUDIO_TEST_DIR)
print(f'Test records: {len(test_recs)}')
test_texts  = transcribe_records(test_recs, desc='Test ASR')
test_labels = [r['label'] for r in test_recs]

del asr_model, asr_processor; torch.cuda.empty_cache()

test_ds = TextDataset(test_texts, test_labels, tokenizer)
test_ld = DataLoader(test_ds, batch_size=BERT_BATCH, shuffle=False, num_workers=2)

preds_all, labels_all = [], []
with torch.no_grad():
    for batch in tqdm(test_ld, desc='Test eval'):
        batch = {k: v.to(device) for k, v in batch.items()}
        out   = bert_best(**batch)
        preds_all.append(out.logits.argmax(-1).cpu().numpy())
        labels_all.append(batch['labels'].cpu().numpy())

preds  = np.concatenate(preds_all)
labels = np.concatenate(labels_all)

print('\n=== Test Results ===')
print(f'Accuracy          : {accuracy_score(labels, preds):.4f}')
print(f'Weighted Accuracy  : {balanced_accuracy_score(labels, preds):.4f}')
print()
print(classification_report(labels, preds, target_names=DUSHA_LABELS, zero_division=0))